In [1]:
# Core libraries
import pandas as pd
import numpy as np
import json

#preprocessing
from sklearn import set_config
set_config(display = 'diagram', transform_output= "pandas")

# Sklearn preprocessing
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Model selection
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import cross_val_score,cross_validate, TimeSeriesSplit
import optuna, optuna_dashboard

#feature selection
from sklearn.inspection import permutation_importance

# pipelines
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.pipeline import make_pipeline, FunctionTransformer

# models
from sklearn.linear_model import ElasticNet #always like to have a linear model for comparison
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats


# Project config
from src.params import *
from src.utils import *
from src.preprocess.cleaning import *
from src.preprocess.features import *

# Plot configuration
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
import warnings
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

1. get data (from local ou BQ)
2. airqual: remove negative values
3. airqual: filter sensors based on thresholds (params.py)
4. airqual: average sensors par city/date
5. merge airqual + weather → NaN apparaissent sur les jours manquants airqual
6. impute gaps ≤ 2j par interpolation linéaire (par city)
7. generate target : pm25 shift(-1) par city
8. log transform target : log(target + 1)
9. feature engineering :
   - lag features : shift(0,1,2,6,12,13,14)
   - mean(shift(12), shift(13), shift(14))
   - std(shift(0)...shift(6))
   - weather features : shift(0)
   - time features : month_sin/cos (date+1), day_of_week (date+1), is_weekend (date+1)
   - city : catégorielle native LightGBM
10. dropna → élimine gaps résiduels + début de série + dernière row


# 0. Load Data
- ingest data from API or from local sources if already downloaded **for the list of cities selected during EDA**

### airqual data

In [2]:
#load airqual
df_airqual = load_data_local(filepath= "../data/raw/airqual.csv", source="airqual")
#remove neg valuues
df_airqual_no_neg = clean_neg_values(df_airqual)

✅ Loaded 17951 rows from ../data/raw/airqual.csv
⚠️  162 aberrant (negative or zero) values found (0.90%)
✅ All negative values removed — 17789 rows remaining


In [3]:
# filter sensors
df_airqual_sensor_filtered = filter_sensors(df= df_airqual_no_neg, max_gap= MAX_GAP, max_q= MAX_Q,
                                     min_bad_month_pct= MIN_BAD_MONTH_PCT, min_coverage_pct= MIN_COVERAGE_PCT)

  [gap filter] 4 sensor(s) flagged — max_gap=30d, max_q75=10.0d
  [coverage filter] 1 sensor(s) flagged — min_cov=70%, max_bad_months=20%
✅ filter_sensors: 4 sensor(s) removed, 22 remaining


In [4]:
#average sensors
df_airqual_col_selected = filter_columns(df= df_airqual_sensor_filtered, col_to_keep=["date", "city", "sensor_id", "pm25_avg"])

In [5]:
df_airqual_averaged = average_sensors(df= df_airqual_col_selected)

✅ Sensors averaged — 4278 rows (city × date)


In [6]:
df_airqual_averaged

,city,date,pm25_avg
0,Berlin,2023-05-01,13.300000
1,Berlin,2023-05-02,10.450000
2,Berlin,2023-05-03,8.456667
3,Berlin,2023-05-04,9.633333
4,Berlin,2023-05-05,10.176667
...,...,...,...
4273,Rome,2025-04-26,4.750000
4274,Rome,2025-04-27,7.500000
4275,Rome,2025-04-28,9.750000
4276,Rome,2025-04-29,8.500000


### Weather data

In [7]:
#load weather data
df_weather = load_data_local(filepath= "../data/raw/weather.csv", source= "weather")

#filter out temp avg data
df_weather_col_filtered = filter_columns(df= df_weather, col_to_remove= ["temp_avg"])

✅ Loaded 4386 rows from ../data/raw/weather.csv


# I. Merge data 

In [8]:
data = merge_source_df(df_airqual= df_airqual_averaged, df_weather= df_weather)

✅ DataFrames merged successfully — 4278/4386 days have airqual data (97.5%)


# I. Impute 1-day gaps


In [9]:
data = single_gaps_imputer(df= data)

total nan before: 108
✅ Imputing successful.1-day gaps before: 108. After: 55


# II. generate and transform target 

In [10]:
data = generate_target(data)

In [11]:
data = target_transform(data)

# III. feature engineering

In [12]:
#nan for ref
data[data["pm25_avg"].isna()].index

Index([  76,  118,  293,  317,  318,  319,  320,  321,  324,  325,  326,  807,
       1779, 1780, 1781, 1782, 1783, 1786, 1787, 1788, 1935, 3241, 3242, 3243,
       3244, 3245, 3248, 3249, 3250, 3257, 3260, 3263, 3270, 3271, 3286, 3312,
       3313, 3316, 3317, 3418, 3419, 3547, 3548, 3549, 3550, 3551, 3601, 3647,
       3654, 3764, 3765, 4043, 4044, 4197, 4270],
      dtype='int64')

In [13]:
# month features
data = month_encoding(data)


In [14]:
#day features
data = day_encoding(data)

In [15]:
#average features
data = average_feature_generation(data)

In [16]:
data = std_feature_generation(data)

In [17]:
data = generate_lag_features(data, shifts= CUSTOM_SHIFTS)

In [18]:
data

,city,date,pm25_avg,temp_min,temp_max,temp_avg,cloud_cover,humidity,precipitation,pressure,...,month_cos,month_sin,dow,is_weekend,lag_avg_14,week_std,lag_1,lag_2,lag_3,lag_7
0,Berlin,2023-05-01,13.300000,4.70,19.37,12.035,0.0,38.0,0.00,1015.0,...,-0.866025,0.500000,0,0,NaN,NaN,13.300000,NaN,NaN,NaN
1,Berlin,2023-05-02,10.450000,9.81,13.29,11.550,75.0,72.0,0.00,1018.0,...,-0.866025,0.500000,1,0,NaN,NaN,10.450000,13.300000,NaN,NaN
2,Berlin,2023-05-03,8.456667,4.80,15.79,10.295,20.0,48.0,0.00,1027.0,...,-0.866025,0.500000,2,0,NaN,NaN,8.456667,10.450000,13.300000,NaN
3,Berlin,2023-05-04,9.633333,5.65,19.37,12.510,0.0,40.0,0.00,1014.0,...,-0.866025,0.500000,3,0,NaN,NaN,9.633333,8.456667,10.450000,NaN
4,Berlin,2023-05-05,10.176667,9.02,20.14,14.580,0.0,47.0,0.11,1008.0,...,-0.866025,0.500000,4,0,NaN,NaN,10.176667,9.633333,8.456667,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4381,Rome,2025-04-26,4.750000,11.98,22.05,17.015,0.0,59.0,0.00,1013.0,...,-0.500000,0.866025,5,1,11.333333,1.307032,4.750000,6.000000,6.000000,7.50
4382,Rome,2025-04-27,7.500000,14.78,22.09,18.435,20.0,50.0,0.86,1017.0,...,-0.500000,0.866025,6,1,10.750000,1.307032,7.500000,4.750000,6.000000,7.50
4383,Rome,2025-04-28,9.750000,14.93,21.71,18.320,20.0,55.0,2.47,1021.0,...,-0.500000,0.866025,0,0,11.305556,1.730332,9.750000,7.500000,4.750000,8.75
4384,Rome,2025-04-29,8.500000,13.10,24.67,18.885,0.0,54.0,4.74,1021.0,...,-0.500000,0.866025,1,0,8.888889,1.692068,8.500000,9.750000,7.500000,6.75
